# 01 — ETL Pipeline · H&M In-Store Movement & Product Placement

**Session 3 — Fully idempotent DuckDB ETL**

---

<details>
<summary><b>📋 CHANGELOG — Click to expand: What changed and why</b></summary>

### 🔒 Idempotency Changes

| # | What changed | Before | After | Why |
|---|---|---|---|---|
| 1 | **Single BASE path variable** | Path written in 6 different cells | One `BASE = Path(...)` at the top | Change folder once — never hunt through cells |
| 2 | **Assert row counts** | Just printed count, kept going | `assert n == 105542` and `assert n == 1371980` | Hard stop if data drifts — catch mistakes at source not 10 cells later |
| 3 | **`CREATE OR REPLACE VIEW`** | Already correct ✅ | Kept as-is | Views overwrite cleanly on every run |
| 4 | **`CREATE OR REPLACE TABLE`** | Already correct ✅ | Kept as-is | Staging tables rebuilt from scratch — never append |
| 5 | **`COPY TO` for all exports** | Already correct ✅ | Kept as-is | Files overwritten — same input → same file every run |
| 6 | **Idempotency receipt (Cell 14)** | No final check existed | Prints all row counts + file sizes | Run twice, compare receipts — must be identical |

### ✨ New Additions (Session 3 full schema)

| # | What was added | Cell | Why |
|---|---|---|---|
| 1 | **CO_PURCHASED staging table** | Cell 9 | Finds article pairs bought together ≥10 times in the same month — core of placement intelligence |
| 2 | **ProductGroup node CSV** | Cell 11 | `neo4j_node_ProductGroup.csv` — needed for IN_GROUP relationships |
| 3 | **Department node CSV** | Cell 11 | `neo4j_node_Department.csv` — needed for BELONGS_TO_DEPT relationships |
| 4 | **StoreSection node CSV** | Cell 11 | `neo4j_node_StoreSection.csv` — needed for IN_SECTION relationships |
| 5 | **IN_GROUP relationship CSV** | Cell 12 | Article → ProductGroup |
| 6 | **BELONGS_TO_DEPT relationship CSV** | Cell 12 | Article → Department |
| 7 | **IN_SECTION relationship CSV** | Cell 12 | Article → StoreSection |
| 8 | **CO_PURCHASED relationship CSV** | Cell 12 | Article ↔ Article co-purchase pairs |

### ✅ Acceptance Test
Run this notebook top to bottom. Run it again. **Cell 14 must show identical numbers both times.**  
If any count differs → a `CREATE` is hiding where `CREATE OR REPLACE` belongs.

</details>

---

**Full schema exported:**
- **Nodes (5):** `Article`, `Customer`, `ProductGroup`, `Department`, `StoreSection`
- **Relationships (5):** `PURCHASED`, `IN_GROUP`, `BELONGS_TO_DEPT`, `IN_SECTION`, `CO_PURCHASED`


In [33]:
# ── 0. Dependencies ───────────────────────────────────────────────────────────
# !pip install duckdb pandas pyarrow
import duckdb, pandas as pd, csv
from pathlib import Path
print(f'DuckDB : {duckdb.__version__}  |  Pandas : {pd.__version__}')

DuckDB : 1.5.2  |  Pandas : 2.2.3


In [ ]:
# ── 1. Paths — edit ONLY this cell if your paths change ──────────────────────
BASE        = Path('C:/Users/prasa/Downloads/H&M')
RAW_DIR     = BASE
PARQUET_DIR = BASE / 'parquet'
NEO4J_DIR = Path('C:/Users/prasa/OneDrive/Desktop/BigData_BI_Capstone/data/clean')

# Idempotent folder creation
PARQUET_DIR.mkdir(parents=True, exist_ok=True)
NEO4J_DIR.mkdir(parents=True, exist_ok=True)

required = ['articles.csv', 'customers.csv', 'transactions_train.csv']
all_ok = True
for f in required:
    p = RAW_DIR / f
    ok = p.exists() and p.stat().st_size > 1000
    size = f'{p.stat().st_size/1e6:.1f} MB' if p.exists() else 'NOT FOUND'
    print(f"  {'✅' if ok else '❌'}  {f:35s}  {size}")
    if not ok: all_ok = False

if not all_ok:
    raise FileNotFoundError('One or more raw files missing')
print('\n  All raw files confirmed ✅')

  ✅  articles.csv                         36.1 MB
  ✅  customers.csv                        207.1 MB
  ✅  transactions_train.csv               3488.0 MB

  All raw files confirmed ✅


In [35]:
# ── 2. DuckDB connection + raw views ─────────────────────────────────────────
# CREATE OR REPLACE VIEW is idempotent — re-run safe
con = duckdb.connect()

con.execute(f"CREATE OR REPLACE VIEW raw_articles     AS SELECT * FROM read_csv_auto('{RAW_DIR}/articles.csv', header=true)")
con.execute(f"CREATE OR REPLACE VIEW raw_customers    AS SELECT * FROM read_csv_auto('{RAW_DIR}/customers.csv', header=true)")
con.execute(f"CREATE OR REPLACE VIEW raw_transactions AS SELECT * FROM read_csv_auto('{RAW_DIR}/transactions_train.csv', header=true)")

print('Views registered. Row counts:')
for v in ['raw_articles', 'raw_customers', 'raw_transactions']:
    n = con.execute(f'SELECT COUNT(*) FROM {v}').fetchone()[0]
    print(f'  {v:25s}  {n:>12,}')

Views registered. Row counts:
  raw_articles                    105,542
  raw_customers                 1,371,980
  raw_transactions             31,788,324


In [36]:
# ── 3. Schema inspection (read-only, safe to re-run) ─────────────────────────
for view in ['raw_articles', 'raw_customers', 'raw_transactions']:
    print(f'\n{"─"*60}  {view.upper()}')
    print(con.execute(f'DESCRIBE {view}').df()[['column_name','column_type']].to_string(index=False))
    print(con.execute(f'SELECT * FROM {view} LIMIT 2').df().to_string())


────────────────────────────────────────────────────────────  RAW_ARTICLES
                 column_name column_type
                  article_id     VARCHAR
                product_code     VARCHAR
                   prod_name     VARCHAR
             product_type_no      BIGINT
           product_type_name     VARCHAR
          product_group_name     VARCHAR
     graphical_appearance_no      BIGINT
   graphical_appearance_name     VARCHAR
           colour_group_code     VARCHAR
           colour_group_name     VARCHAR
   perceived_colour_value_id      BIGINT
 perceived_colour_value_name     VARCHAR
  perceived_colour_master_id      BIGINT
perceived_colour_master_name     VARCHAR
               department_no      BIGINT
             department_name     VARCHAR
                  index_code     VARCHAR
                  index_name     VARCHAR
              index_group_no      BIGINT
            index_group_name     VARCHAR
                  section_no      BIGINT
                sectio

In [37]:
# ── 4a. Null quality checks ───────────────────────────────────────────────────
print('=== ARTICLES — null counts ===')
con.execute("""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) - COUNT(article_id)        AS null_article_id,
           COUNT(*) - COUNT(prod_name)         AS null_prod_name,
           COUNT(*) - COUNT(colour_group_name) AS null_colour,
           COUNT(*) - COUNT(detail_desc)       AS null_desc
    FROM raw_articles
""").df().T.rename(columns={0:'count'}).pipe(print)

print('\n=== CUSTOMERS — null counts ===')
con.execute("""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) - COUNT(customer_id)            AS null_customer_id,
           COUNT(*) - COUNT(age)                    AS null_age,
           COUNT(*) - COUNT(club_member_status)     AS null_member_status,
           COUNT(*) - COUNT(fashion_news_frequency) AS null_news_freq
    FROM raw_customers
""").df().T.rename(columns={0:'count'}).pipe(print)

print('\n=== TRANSACTIONS — null counts ===')
con.execute("""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) - COUNT(t_dat)       AS null_date,
           COUNT(*) - COUNT(customer_id) AS null_customer_id,
           COUNT(*) - COUNT(article_id)  AS null_article_id,
           COUNT(*) - COUNT(price)       AS null_price
    FROM raw_transactions
""").df().T.rename(columns={0:'count'}).pipe(print)

=== ARTICLES — null counts ===
                  count
total_rows       105542
null_article_id       0
null_prod_name        0
null_colour           0
null_desc           416

=== CUSTOMERS — null counts ===
                      count
total_rows          1371980
null_customer_id          0
null_age              15861
null_member_status     6062
null_news_freq        16009

=== TRANSACTIONS — null counts ===
                     count
total_rows        31788324
null_date                0
null_customer_id         0
null_article_id          0
null_price               0


In [38]:
# ── 4b. Value distributions ───────────────────────────────────────────────────
print('=== Sales channel split ===')
con.execute("""
    SELECT sales_channel_id, COUNT(*) AS tx_count,
           ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(),2) AS pct
    FROM raw_transactions GROUP BY 1 ORDER BY 1
""").df().pipe(print)

print('\n=== Customer age bands ===')
con.execute("""
    SELECT CASE WHEN age IS NULL THEN 'Unknown'
                WHEN age < 20 THEN 'Under 20'
                WHEN age BETWEEN 20 AND 29 THEN '20-29'
                WHEN age BETWEEN 30 AND 39 THEN '30-39'
                WHEN age BETWEEN 40 AND 49 THEN '40-49'
                WHEN age BETWEEN 50 AND 59 THEN '50-59'
                ELSE '60+' END AS age_band,
           COUNT(*) AS customer_count
    FROM raw_customers GROUP BY 1 ORDER BY 2 DESC
""").df().pipe(print)

print('\n=== Top 10 product types ===')
con.execute("""
    SELECT product_type_name, COUNT(*) AS article_count
    FROM raw_articles GROUP BY 1 ORDER BY 2 DESC LIMIT 10
""").df().pipe(print)

=== Sales channel split ===
   sales_channel_id  tx_count   pct
0                 1   9408462  29.6
1                 2  22379862  70.4

=== Customer age bands ===
   age_band  customer_count
0     20-29          528358
1     30-39          234068
2     50-59          226242
3     40-49          204118
4       60+           91750
5  Under 20           71583
6   Unknown           15861

=== Top 10 product types ===
  product_type_name  article_count
0          Trousers          11169
1             Dress          10362
2           Sweater           9302
3           T-shirt           7904
4               Top           4155
5            Blouse           3979
6            Jacket           3940
7            Shorts           3939
8             Shirt           3405
9          Vest top           2991


In [39]:
# ── 5. Clean ARTICLES ─────────────────────────────────────────────────────────
# IDEMPOTENT: CREATE OR REPLACE TABLE — overwrites on every run, never appends
con.execute("""
    CREATE OR REPLACE TABLE clean_articles AS
    SELECT
        CAST(article_id AS VARCHAR)                              AS article_id,
        CAST(product_code AS VARCHAR)                            AS product_code,
        TRIM(prod_name)                                          AS prod_name,
        COALESCE(TRIM(product_type_name),  'Unknown')            AS product_type_name,
        COALESCE(TRIM(product_group_name), 'Unknown')            AS product_group_name,
        COALESCE(TRIM(graphical_appearance_name), 'Unknown')     AS graphical_appearance_name,
        COALESCE(TRIM(colour_group_name),  'Unknown')            AS colour_group_name,
        COALESCE(TRIM(perceived_colour_value_name), 'Unknown')   AS perceived_colour_value,
        COALESCE(TRIM(perceived_colour_master_name), 'Unknown')  AS perceived_colour_master,
        COALESCE(TRIM(department_name),    'Unknown')            AS department_name,
        COALESCE(TRIM(index_name),         'Unknown')            AS index_name,
        COALESCE(TRIM(index_group_name),   'Unknown')            AS index_group_name,
        COALESCE(TRIM(section_name),       'Unknown')            AS section_name,
        COALESCE(TRIM(garment_group_name), 'Unknown')            AS garment_group_name,
        COALESCE(TRIM(detail_desc), 'No description available')  AS detail_desc,
        CASE
            WHEN index_group_name ILIKE '%ladieswear%' THEN 'Womenswear'
            WHEN index_group_name ILIKE '%menswear%'   THEN 'Menswear'
            WHEN index_group_name ILIKE '%divided%'    THEN 'Young Fashion'
            WHEN index_group_name ILIKE '%children%'   THEN 'Kidswear'
            WHEN index_group_name ILIKE '%sport%'      THEN 'Sport'
            ELSE 'Other'
        END AS store_section
    FROM raw_articles
    WHERE article_id IS NOT NULL
""")
n = con.execute('SELECT COUNT(*) FROM clean_articles').fetchone()[0]
print(f'clean_articles: {n:,} rows  (expected 105,542)')
assert n == 105542, f'IDEMPOTENCY FAIL: expected 105542 got {n}'
con.execute('SELECT * FROM clean_articles LIMIT 3').df()

clean_articles: 105,542 rows  (expected 105,542)


,article_id,product_code,prod_name,product_type_name,product_group_name,graphical_appearance_name,colour_group_name,perceived_colour_value,perceived_colour_master,department_name,index_name,index_group_name,section_name,garment_group_name,detail_desc,store_section
0,0108775015,0108775,Strap top,Vest top,Garment Upper body,Solid,Black,Dark,Black,Jersey Basic,Ladieswear,Ladieswear,Womens Everyday Basics,Jersey Basic,Jersey top with narrow shoulder straps.,Womenswear
1,0108775044,0108775,Strap top,Vest top,Garment Upper body,Solid,White,Light,White,Jersey Basic,Ladieswear,Ladieswear,Womens Everyday Basics,Jersey Basic,Jersey top with narrow shoulder straps.,Womenswear
2,0108775051,0108775,Strap top (1),Vest top,Garment Upper body,Stripe,Off White,Dusty Light,White,Jersey Basic,Ladieswear,Ladieswear,Womens Everyday Basics,Jersey Basic,Jersey top with narrow shoulder straps.,Womenswear


In [40]:
# ── 6. Clean CUSTOMERS ────────────────────────────────────────────────────────
# IDEMPOTENT: CREATE OR REPLACE TABLE — overwrites on every run, never appends
con.execute("""
    CREATE OR REPLACE TABLE clean_customers AS
    SELECT
        TRIM(customer_id)                                        AS customer_id,
        CAST(COALESCE(FN, 0) AS INTEGER)                         AS fn_flag,
        CAST(COALESCE(Active, 0) AS INTEGER)                     AS active_flag,
        UPPER(COALESCE(TRIM(club_member_status), 'UNKNOWN'))     AS club_member_status,
        UPPER(COALESCE(TRIM(fashion_news_frequency), 'NONE'))    AS fashion_news_frequency,
        CAST(COALESCE(age, 29) AS INTEGER)                       AS age,
        TRIM(postal_code)                                        AS postal_code,
        CASE
            WHEN COALESCE(age,29) < 20              THEN 'Under_20'
            WHEN COALESCE(age,29) BETWEEN 20 AND 29 THEN '20s'
            WHEN COALESCE(age,29) BETWEEN 30 AND 39 THEN '30s'
            WHEN COALESCE(age,29) BETWEEN 40 AND 49 THEN '40s'
            WHEN COALESCE(age,29) BETWEEN 50 AND 59 THEN '50s'
            ELSE '60_plus'
        END AS age_band
    FROM raw_customers
    WHERE customer_id IS NOT NULL
""")
n = con.execute('SELECT COUNT(*) FROM clean_customers').fetchone()[0]
print(f'clean_customers: {n:,} rows  (expected 1,371,980)')
assert n == 1371980, f'IDEMPOTENCY FAIL: expected 1371980 got {n}'
con.execute('SELECT * FROM clean_customers LIMIT 3').df()

clean_customers: 1,371,980 rows  (expected 1,371,980)


,customer_id,fn_flag,active_flag,club_member_status,fashion_news_frequency,age,postal_code,age_band
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0,0,ACTIVE,NONE,49,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...,40s
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0,0,ACTIVE,NONE,25,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...,20s
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0,0,ACTIVE,NONE,24,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...,20s


In [41]:
# ── 7. Clean TRANSACTIONS — In-Store only ─────────────────────────────────────
# Filtering to In-Store channel only (sales_channel_id = 1)
# Reason: project is about in-store placement — online data is irrelevant
# Reduces dataset from 27M → ~9-10M rows — manageable on local Neo4j
# IDEMPOTENT: CREATE OR REPLACE TABLE — overwrites on every run

con.execute("""
    CREATE OR REPLACE TABLE clean_transactions AS
    SELECT
        CAST(t.t_dat AS DATE)                    AS tx_date,
        STRFTIME(CAST(t.t_dat AS DATE), '%Y-%m') AS year_month,
        TRIM(t.customer_id)                      AS customer_id,
        CAST(t.article_id AS VARCHAR)            AS article_id,
        ROUND(CAST(t.price AS DOUBLE), 4)        AS price,
        CAST(t.sales_channel_id AS INTEGER)      AS sales_channel_id,
        'In_Store'                               AS channel_label
    FROM raw_transactions t
    INNER JOIN clean_customers c ON TRIM(t.customer_id) = c.customer_id
    INNER JOIN clean_articles  a ON CAST(t.article_id AS VARCHAR) = a.article_id
    WHERE t.t_dat    IS NOT NULL
      AND t.price    IS NOT NULL
      AND t.price    > 0
      AND CAST(t.sales_channel_id AS INTEGER) = 1
""")
n = con.execute('SELECT COUNT(*) FROM clean_transactions').fetchone()[0]
print(f'clean_transactions (In-Store only): {n:,} rows')
print(f'Channel check:')
con.execute("""
    SELECT channel_label, COUNT(*) AS count
    FROM clean_transactions GROUP BY 1
""").df().pipe(print)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

clean_transactions (In-Store only): 9,408,462 rows
Channel check:
  channel_label    count
0      In_Store  9408462


In [42]:
# ── 8. Sanity checks ──────────────────────────────────────────────────────────
print('=== Price range ===')
con.execute("""
    SELECT ROUND(MIN(price),4) AS min_price, ROUND(MAX(price),4) AS max_price,
           ROUND(AVG(price),4) AS avg_price, ROUND(MEDIAN(price),4) AS median_price
    FROM clean_transactions
""").df().pipe(print)

print('\n=== Channel split ===')
con.execute("""
    SELECT channel_label, COUNT(*) AS tx_count,
           ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(),2) AS pct
    FROM clean_transactions GROUP BY 1 ORDER BY 2 DESC
""").df().pipe(print)

print('\n=== Monthly trend (first 6) ===')
con.execute("""
    SELECT year_month, COUNT(*) AS tx_count
    FROM clean_transactions GROUP BY 1 ORDER BY 1 LIMIT 6
""").df().pipe(print)

=== Price range ===
   min_price  max_price  avg_price  median_price
0        0.0     0.5324     0.0229        0.0186

=== Channel split ===
  channel_label  tx_count    pct
0      In_Store   9408462  100.0

=== Monthly trend (first 6) ===
  year_month  tx_count
0    2018-09    207326
1    2018-10    422989
2    2018-11    360007
3    2018-12    442405
4    2019-01    330749
5    2019-02    376081


In [43]:
# ── 9. Build CO_PURCHASED staging table ───────────────────────────────────────
# IDEMPOTENT: CREATE OR REPLACE TABLE — deterministic rebuild
# Same input (clean_transactions) → same output (pairs + counts) every run
print('Building CO_PURCHASED pairs — takes 3-5 minutes...')
con.execute("""
    CREATE OR REPLACE TABLE co_purchased AS
    SELECT
        t1.article_id  AS article1Id,
        t2.article_id  AS article2Id,
        COUNT(*)       AS timesBoughtTogether,
        ROUND(COUNT(*) * 1.0 /
            (SELECT COUNT(DISTINCT customer_id) FROM clean_transactions), 6)
                       AS supportScore
    FROM clean_transactions t1
    JOIN clean_transactions t2
        ON  t1.customer_id = t2.customer_id
        AND t1.year_month  = t2.year_month
        AND t1.article_id  < t2.article_id
    GROUP BY 1, 2
    HAVING COUNT(*) >= 10
    ORDER BY 3 DESC
""")
n = con.execute('SELECT COUNT(*) FROM co_purchased').fetchone()[0]
print(f'  ✅  co_purchased pairs: {n:,}')
con.execute('SELECT * FROM co_purchased LIMIT 5').df()

Building CO_PURCHASED pairs — takes 3-5 minutes...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  ✅  co_purchased pairs: 109,684


,article1Id,article2Id,timesBoughtTogether,supportScore
0,0561445005,0678342001,45642,0.061920
1,0501616007,0507909001,7684,0.010424
2,0570002001,0570002002,4268,0.005790
3,0372860001,0372860002,3088,0.004189
4,0111586001,0111593001,2638,0.003579


In [44]:
# ── 10. Export PARQUET files ──────────────────────────────────────────────────
# IDEMPOTENT: COPY TO overwrites existing file — no accumulation ever
for table, path in [
    ('clean_articles',     PARQUET_DIR / 'articles.parquet'),
    ('clean_customers',    PARQUET_DIR / 'customers.parquet'),
    ('clean_transactions', PARQUET_DIR / 'transactions.parquet'),
]:
    con.execute(f"COPY {table} TO '{path}' (FORMAT PARQUET, COMPRESSION SNAPPY)")
    print(f"  ✅  {path.name:<35}  {path.stat().st_size/1e6:.1f} MB")

  ✅  articles.parquet                     5.7 MB
  ✅  customers.parquet                    168.3 MB


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  ✅  transactions.parquet                 294.9 MB


In [45]:
# ── 11. Export NODE CSVs ──────────────────────────────────────────────────────
# IDEMPOTENT: COPY TO overwrites — same input → same file every run

node_exports = [
    # (label, query, expected_count)
    ('Article', f"""
        SELECT article_id AS articleId, prod_name AS name,
               product_type_name AS productType, product_group_name AS productGroup,
               colour_group_name AS colourGroup, perceived_colour_master AS colour,
               department_name AS department, index_group_name AS indexGroup,
               section_name AS section, garment_group_name AS garmentGroup,
               store_section AS storeSection, detail_desc AS description
        FROM clean_articles
    """, None),
    ('Customer', f"""
        SELECT customer_id AS customerId, age, age_band AS ageBand,
               club_member_status AS memberStatus,
               fashion_news_frequency AS newsFrequency,
               fn_flag AS fnFlag, active_flag AS activeFlag,
               postal_code AS postalCode
        FROM clean_customers
    """, None),
    ('ProductGroup', """
        SELECT DISTINCT product_group_name AS productGroupId,
                        product_group_name AS name
        FROM clean_articles WHERE product_group_name IS NOT NULL
    """, None),
    ('Department', """
        SELECT DISTINCT department_name AS departmentId,
                        department_name AS name,
                        index_group_name AS indexGroup
        FROM clean_articles WHERE department_name IS NOT NULL
    """, None),
    ('StoreSection', """
        SELECT DISTINCT store_section AS storeSectionId,
                        store_section AS name
        FROM clean_articles WHERE store_section IS NOT NULL
    """, None),
]

for label, query, _ in node_exports:
    path = NEO4J_DIR / f'neo4j_node_{label}.csv'
    con.execute(f"COPY ({query}) TO '{path}' (FORMAT CSV, HEADER true)")
    n = con.execute(f"SELECT COUNT(*) FROM ({query})").fetchone()[0]
    print(f'  ✅  neo4j_node_{label}.csv  {n:>10,} rows')

  ✅  neo4j_node_Article.csv     105,542 rows
  ✅  neo4j_node_Customer.csv   1,371,980 rows
  ✅  neo4j_node_ProductGroup.csv          19 rows
  ✅  neo4j_node_Department.csv         263 rows
  ✅  neo4j_node_StoreSection.csv           5 rows


In [46]:
# ── 12. Export RELATIONSHIP CSVs ──────────────────────────────────────────────
# IDEMPOTENT: COPY TO overwrites — same input → same file every run

rel_exports = [
    ('PURCHASED', """
        SELECT customer_id AS customerId, article_id AS articleId,
               tx_date AS txDate, year_month AS yearMonth,
               price, channel_label AS channel
        FROM clean_transactions
    """),
    ('IN_GROUP', """
        SELECT DISTINCT article_id AS articleId,
                        product_group_name AS productGroupId
        FROM clean_articles WHERE product_group_name IS NOT NULL
    """),
    ('BELONGS_TO_DEPT', """
        SELECT DISTINCT article_id AS articleId,
                        department_name AS departmentId
        FROM clean_articles WHERE department_name IS NOT NULL
    """),
    ('IN_SECTION', """
        SELECT DISTINCT article_id AS articleId,
                        store_section AS storeSectionId
        FROM clean_articles WHERE store_section IS NOT NULL
    """),
    ('CO_PURCHASED', 'SELECT * FROM co_purchased'),
]

for rel_type, query in rel_exports:
    path = NEO4J_DIR / f'neo4j_rel_{rel_type}.csv'
    con.execute(f"COPY ({query}) TO '{path}' (FORMAT CSV, HEADER true)")
    n = con.execute(f'SELECT COUNT(*) FROM ({query})').fetchone()[0]
    print(f'  ✅  neo4j_rel_{rel_type}.csv  {n:>10,} rows')

  ✅  neo4j_rel_PURCHASED.csv   9,408,462 rows
  ✅  neo4j_rel_IN_GROUP.csv     105,542 rows
  ✅  neo4j_rel_BELONGS_TO_DEPT.csv     105,542 rows
  ✅  neo4j_rel_IN_SECTION.csv     105,542 rows
  ✅  neo4j_rel_CO_PURCHASED.csv     109,684 rows


In [47]:
# ── 13. CSV header preview ────────────────────────────────────────────────────
for fname in sorted(NEO4J_DIR.glob('neo4j_*.csv')):
    with open(fname) as f:
        reader = csv.reader(f)
        headers = next(reader)
        row1    = next(reader)
    print(f'\n{fname.name}')
    print(f'  cols : {headers}')
    print(f'  row1 : {row1}')


neo4j_node_Article.csv
  cols : ['articleId', 'name', 'productType', 'productGroup', 'colourGroup', 'colour', 'department', 'indexGroup', 'section', 'garmentGroup', 'storeSection', 'description']
  row1 : ['0108775015', 'Strap top', 'Vest top', 'Garment Upper body', 'Black', 'Black', 'Jersey Basic', 'Ladieswear', 'Womens Everyday Basics', 'Jersey Basic', 'Womenswear', 'Jersey top with narrow shoulder straps.']

neo4j_node_Customer.csv
  cols : ['customerId', 'age', 'ageBand', 'memberStatus', 'newsFrequency', 'fnFlag', 'activeFlag', 'postalCode']
  row1 : ['00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657', '49', '40s', 'ACTIVE', 'NONE', '0', '0', '52043ee2162cf5aa7ee79974281641c6f11a68d276429a91f8ca0d4b6efa8100']

neo4j_node_Department.csv
  cols : ['departmentId', 'name', 'indexGroup']
  row1 : ['Belts', 'Belts', 'Ladieswear']

neo4j_node_ProductGroup.csv
  cols : ['productGroupId', 'name']
  row1 : ['Underwear', 'Underwear']

neo4j_node_StoreSection.csv
  cols : ['st

In [48]:
# ── 14. IDEMPOTENCY RECEIPT — DuckDB side ────────────────────────────────────
# Run notebook twice. These numbers must be identical both times.
# If any number differs, CREATE OR REPLACE is broken somewhere above.
print('=' * 65)
print('  ETL IDEMPOTENCY RECEIPT — counts must match on every re-run')
print('=' * 65)

staging_checks = [
    ('clean_articles',     105542,  'rows'),
    ('clean_customers',    1371980, 'rows'),
    ('clean_transactions', None,    'rows'),
    ('co_purchased',       None,    'pairs'),
]
all_ok = True
for table, expected, unit in staging_checks:
    actual = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    if expected and actual != expected:
        print(f'  ❌  {table:<25}  {actual:>12,} {unit}  (expected {expected:,})')
        all_ok = False
    else:
        print(f'  ✅  {table:<25}  {actual:>12,} {unit}')

print('\n  OUTPUT FILES:')
for p in sorted((NEO4J_DIR).glob('neo4j_*.csv')):
    size = p.stat().st_size / 1e6
    print(f'  ✅  {p.name:<45}  {size:>8.1f} MB')
for p in sorted(PARQUET_DIR.glob('*.parquet')):
    size = p.stat().st_size / 1e6
    print(f'  ✅  {p.name:<45}  {size:>8.1f} MB')

print()
print('  ✅  ETL complete — all files ready for 02_graph_load.ipynb')
con.close()

  ETL IDEMPOTENCY RECEIPT — counts must match on every re-run
  ✅  clean_articles                  105,542 rows
  ✅  clean_customers               1,371,980 rows
  ✅  clean_transactions            9,408,462 rows
  ✅  co_purchased                    109,684 pairs

  OUTPUT FILES:
  ✅  neo4j_node_Article.csv                             29.2 MB
  ✅  neo4j_node_Customer.csv                           213.4 MB
  ✅  neo4j_node_Department.csv                           0.0 MB
  ✅  neo4j_node_ProductGroup.csv                         0.0 MB
  ✅  neo4j_node_StoreSection.csv                         0.0 MB
  ✅  neo4j_rel_BELONGS_TO_DEPT.csv                       2.7 MB
  ✅  neo4j_rel_CO_PURCHASED.csv                          3.6 MB
  ✅  neo4j_rel_IN_GROUP.csv                              2.9 MB
  ✅  neo4j_rel_IN_SECTION.csv                            2.3 MB
  ✅  neo4j_rel_PURCHASED.csv                          1043.8 MB
  ✅  articles.parquet                                    5.7 MB
  ✅  customers.p